# Tutorial 13: JUMP Cell Painting Morphological Embeddings

This notebook demonstrates how to load and analyse **JUMP Cell Painting**
morphological embeddings. The [JUMP](https://jump-cellpainting.broadinstitute.org/)
(Joint Undertaking for Morphological Profiling) consortium performed
large-scale CRISPR perturbation screens in U2OS cells, measuring over
3,000 image-derived cellular morphology features via Cell Painting.

We use the **batch-corrected CRISPR embeddings** from the
[Cell Painting Gallery](https://registry.opendata.aws/cellpainting-gallery/):

- **51,185 wells** across 148 plates
- **7,977 unique gene perturbations** (identified by JCP2022 IDs)
- **259-dimensional** PCA-reduced embeddings after Harmony batch correction

### What you'll learn

1. Load JUMP CRISPR embeddings from parquet
2. Map JCP2022 identifiers to gene names with `broad-babel`
3. Separate treatments from negative controls
4. Compute per-gene mean embeddings (pseudobulk)
5. Find morphologically similar gene perturbations
6. Visualise the morphological landscape with PCA
7. Store embeddings as AnnData for downstream analysis

### Requirements

```bash
pip install broad-babel pyarrow
```

Download the embeddings first:
```bash
mkdir -p data/JUMP
wget -O data/JUMP/profiles_wellpos_cc_var_mad_outlier_featselect_sphering_harmony_PCA_corrected.parquet \
  "https://cellpainting-gallery.s3.amazonaws.com/cpg0016-jump-assembled/source_all/workspace/profiles_assembled/CRISPR/v1.0a/profiles_wellpos_cc_var_mad_outlier_featselect_sphering_harmony_PCA_corrected.parquet"
```

In [ ]:
import logging

logging.basicConfig(level=logging.INFO)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

## 1. Load the JUMP CRISPR embeddings

The parquet file contains well-level embeddings (259 PCA components)
with metadata columns for source, plate, well, and JCP2022 perturbation ID.

In [ ]:
JUMP_DIR = Path("../../data/datasets/JUMP")
PARQUET_FILE = JUMP_DIR / "profiles_wellpos_cc_var_mad_outlier_featselect_sphering_harmony_PCA_corrected.parquet"

df = pd.read_parquet(PARQUET_FILE)
print(f"Shape: {df.shape}")
df.head()

In [ ]:
meta_cols = [c for c in df.columns if c.startswith("Metadata_")]
feat_cols = [c for c in df.columns if not c.startswith("Metadata_")]

print(f"Metadata columns ({len(meta_cols)}): {meta_cols}")
print(f"Embedding dimensions: {len(feat_cols)}")
print(f"Unique perturbations: {df['Metadata_JCP2022'].nunique():,}")
print(f"Unique plates: {df['Metadata_Plate'].nunique()}")

## 2. Map JCP2022 IDs to gene names

The perturbations are identified by JCP2022 codes. We use
[broad-babel](https://pypi.org/project/broad-babel/) to translate
them to standard gene symbols.

In [ ]:
from broad_babel.query import get_mapper

mapper = get_mapper(
    query="crispr",
    input_column="plate_type",
    output_columns="JCP2022,standard_key",
)
print(f"Mapper contains {len(mapper):,} JCP2022 -> gene mappings")
print("Sample:", dict(list(mapper.items())[:5]))

In [ ]:
df["gene"] = df["Metadata_JCP2022"].map(mapper)

mapped = df["gene"].notna().sum()
total = len(df)
print(f"Mapped {mapped:,} / {total:,} wells ({100*mapped/total:.1f}%)")
print(f"Unmapped JCP IDs: {df.loc[df['gene'].isna(), 'Metadata_JCP2022'].unique()[:10]}")

## 3. Separate treatments from negative controls

JCP2022_800001 and JCP2022_800002 are non-targeting negative controls.

In [ ]:
NEGCON_IDS = {"JCP2022_800001", "JCP2022_800002"}

df["is_control"] = df["Metadata_JCP2022"].isin(NEGCON_IDS)

n_ctrl = df["is_control"].sum()
n_trt = (~df["is_control"]).sum()
print(f"Negative controls: {n_ctrl:,} wells")
print(f"Treatments: {n_trt:,} wells")
print(f"Unique treated genes: {df.loc[~df['is_control'], 'Metadata_JCP2022'].nunique():,}")

## 4. Embedding distribution

Compare the distribution of embedding magnitudes between
controls and treatments.

In [ ]:
X = df[feat_cols].values.astype(np.float32)

norms = np.linalg.norm(X, axis=1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(norms[df["is_control"].values], bins=50, alpha=0.7, label="Controls", density=True)
axes[0].hist(norms[~df["is_control"].values], bins=50, alpha=0.7, label="Treatments", density=True)
axes[0].set_xlabel("L2 norm of embedding")
axes[0].set_ylabel("Density")
axes[0].set_title("Embedding magnitude: controls vs treatments")
axes[0].legend()

axes[1].hist(X[:, 0], bins=80, alpha=0.7, color="steelblue")
axes[1].set_xlabel("X_1 (first PC)")
axes[1].set_ylabel("Count")
axes[1].set_title("Distribution of first principal component")

plt.tight_layout()
plt.show()

## 5. Per-gene mean embeddings (pseudobulk)

Aggregate replicate wells per perturbation to get a single
embedding per gene.

In [ ]:
trt = df[~df["is_control"]].copy()

gene_embs = trt.groupby("Metadata_JCP2022")[feat_cols].mean()
gene_embs["gene"] = gene_embs.index.map(mapper)

print(f"Per-gene embeddings: {gene_embs.shape[0]:,} genes x {len(feat_cols)} dims")
gene_embs.head()

## 6. Find morphologically similar perturbations

Given a gene of interest, find the perturbations with the
most similar morphological profile using cosine similarity.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

X_genes = gene_embs[feat_cols].values.astype(np.float32)
gene_labels = gene_embs["gene"].values
jcp_ids = gene_embs.index.values

sim_matrix = cosine_similarity(X_genes)
print(f"Similarity matrix: {sim_matrix.shape}")

In [ ]:
def find_similar_genes(query_gene, gene_labels, sim_matrix, top_k=10):
    """Find the top-k most similar gene perturbations."""
    matches = np.where(gene_labels == query_gene)[0]
    if len(matches) == 0:
        jcp_match = np.where(jcp_ids == query_gene)[0]
        if len(jcp_match) == 0:
            print(f"Gene '{query_gene}' not found.")
            return None
        idx = jcp_match[0]
    else:
        idx = matches[0]

    sims = sim_matrix[idx]
    top_idx = np.argsort(sims)[::-1][1 : top_k + 1]

    results = pd.DataFrame({
        "gene": gene_labels[top_idx],
        "jcp_id": jcp_ids[top_idx],
        "cosine_sim": sims[top_idx],
    })
    return results


query = "TP53"
similar = find_similar_genes(query, gene_labels, sim_matrix, top_k=15)
if similar is not None:
    print(f"\nTop 15 genes most similar to {query} (morphological profile):")
    print(similar.to_string(index=False))

## 7. PCA visualisation of the morphological landscape

Project the per-gene embeddings to 2D for visualisation.
We highlight a few genes of interest.

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
coords = pca.fit_transform(X_genes)

fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(coords[:, 0], coords[:, 1], s=3, alpha=0.3, c="grey", label="All genes")

highlight_genes = ["TP53", "BRCA1", "EGFR", "KRAS", "MYC", "PLK1", "CDK2"]
colors = plt.cm.tab10(np.linspace(0, 1, len(highlight_genes)))

for gene, color in zip(highlight_genes, colors):
    mask = gene_labels == gene
    if mask.any():
        ax.scatter(coords[mask, 0], coords[mask, 1], s=80, c=[color],
                   edgecolors="black", linewidths=0.5, label=gene, zorder=5)

ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)")
ax.set_title("JUMP CRISPR morphological landscape (per-gene mean embeddings)")
ax.legend(markerscale=1.5, fontsize=9)
plt.tight_layout()
plt.show()

## 8. Cosine similarity heatmap for selected genes

Visualise pairwise morphological similarity for a set of
cancer-relevant genes.

In [ ]:
genes_of_interest = [
    "TP53", "BRCA1", "BRCA2", "EGFR", "KRAS", "MYC",
    "PLK1", "CDK2", "CDK4", "RB1", "PTEN", "APC",
    "ATM", "ATR", "CHEK1", "CHEK2",
]

mask = np.isin(gene_labels, genes_of_interest)
found_genes = gene_labels[mask]
print(f"Found {mask.sum()} / {len(genes_of_interest)} genes")

sub_X = X_genes[mask]
sub_sim = cosine_similarity(sub_X)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    sub_sim,
    xticklabels=found_genes,
    yticklabels=found_genes,
    cmap="RdBu_r",
    vmin=-1,
    vmax=1,
    square=True,
    ax=ax,
)
ax.set_title("Pairwise morphological similarity (JUMP CRISPR)")
plt.tight_layout()
plt.show()

## 9. Store as AnnData

Convert the per-gene embeddings into an AnnData object for
integration with other `embpy` workflows.

In [ ]:
import anndata as ad

gene_embs_clean = gene_embs.dropna(subset=["gene"])

adata = ad.AnnData(
    X=gene_embs_clean[feat_cols].values.astype(np.float32),
    obs=pd.DataFrame({
        "jcp_id": gene_embs_clean.index,
        "gene": gene_embs_clean["gene"].values,
    }).set_index("gene"),
)
adata.var_names = [f"JUMP_PC{i+1}" for i in range(len(feat_cols))]

print(adata)
print(f"\nFirst 5 genes: {adata.obs_names[:5].tolist()}")

In [ ]:
adata.obsm["X_pca"] = PCA(n_components=50).fit_transform(adata.X)
print(f"Stored PCA(50) in adata.obsm['X_pca']: {adata.obsm['X_pca'].shape}")
adata

## 10. Replicate reproducibility

Assess how reproducible the morphological signal is across
replicate wells for the same perturbation.

In [ ]:
n_sample = 200
rng = np.random.default_rng(42)
sampled_jcps = rng.choice(
    trt["Metadata_JCP2022"].unique(), size=min(n_sample, trt["Metadata_JCP2022"].nunique()), replace=False
)

rep_corrs = []
for jcp in sampled_jcps:
    wells = trt.loc[trt["Metadata_JCP2022"] == jcp, feat_cols].values
    if len(wells) >= 2:
        sims = cosine_similarity(wells)
        upper = sims[np.triu_indices_from(sims, k=1)]
        rep_corrs.append(upper.mean())

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(rep_corrs, bins=50, edgecolor="white", color="steelblue")
ax.axvline(np.median(rep_corrs), color="red", linestyle="--", label=f"Median = {np.median(rep_corrs):.3f}")
ax.set_xlabel("Mean pairwise cosine similarity (replicates)")
ax.set_ylabel("Number of perturbations")
ax.set_title("Replicate reproducibility (JUMP CRISPR)")
ax.legend()
plt.tight_layout()
plt.show()

## Summary

This tutorial showed how to:

- **Load** JUMP Cell Painting CRISPR embeddings (259-dim, Harmony + PCA corrected)
- **Map** JCP2022 IDs to gene symbols with `broad-babel`
- **Separate** treatments from negative controls
- **Aggregate** per-gene mean embeddings across replicate wells
- **Find** morphologically similar gene perturbations via cosine similarity
- **Visualise** the morphological landscape with PCA and heatmaps
- **Store** embeddings as AnnData for integration with `embpy`

### Data source

Batch-corrected CRISPR embeddings from the
[Cell Painting Gallery](https://registry.opendata.aws/cellpainting-gallery/)
release (cpg0016, CRISPR v1.0a). Profile index:
[jump-cellpainting/datasets](https://github.com/jump-cellpainting/datasets).

### References

- Chandrasekaran et al., *Nature Methods* (2025). [JUMP Morphmap](https://github.com/jump-cellpainting/2025_Chandrasekaran_NatureMethods_Morphmap)
- Cimini et al., *Nature Protocols* (2023). Cell Painting protocol.